# L6 - Service Map, Error Tracking & SLOs

## What You'll Learn

- How Datadog auto-builds a **Service Map** of the multi-agent system from the spans
- How **Error Tracking** groups failures (gateway 403s, timeouts, tool errors) into issues
- How to define an **SLO** that turns those metrics into a reliability commitment, and
  how to read the **error budget** it tracks

## Overview

Notebooks 1-4 traced, measured, and evaluated the agents. This notebook turns that span
data into three operational views Datadog derives largely on its own - a dependency map,
grouped error issues, and a reliability target - so you can see health at a glance.

## Architecture

```
Spans from all agents (trace-agent sidecar + Harness OTLP)
        |
        v
   Datadog APM
    |- Service Map     -> auto topology: Harness -> Gateway -> Lambda -> data stores
    |- Error Tracking  -> error spans grouped into issues (403s, timeouts, tool errors)
    +- SLOs            -> reliability target + error budget on the anycompany.* metrics
```

## Steps

1. Setup - Datadog config + a span-search helper
2. Generate sample traffic (so the views have fresh data)
3. Service Map - the auto-built dependency topology
4. Error Tracking - group failures into issues
5. SLOs - a reliability target and error budget on the agents
6. Cleanup


## License Details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- **L1-L3 completed** - agents deployed and emitting spans (sidecar + Harness OTLP)
- **Notebook 2 completed** - Step 5's SLO is built on the `anycompany.*` custom metrics
  that nb2 submits
- App key with `apm_read` (span search). Step 5's SLO create also needs `slos_write`;
  without it the notebook prints the SLO definition to create in the UI instead.
  Scopes are chosen when the key is created — the full list is in
  `l3-orchestration/1_setup_resources.ipynb` Step 6.


## Step 1: Setup

In [ ]:
%pip install --quiet datadog-api-client boto3 --upgrade

In [ ]:
import os, time, json, uuid
from datetime import datetime, timedelta, timezone
from collections import defaultdict
import boto3
from botocore.config import Config
from datadog_api_client import Configuration, ApiClient
from datadog_api_client.v2.api.spans_api import SpansApi
from datadog_api_client.v2.model.spans_list_request import SpansListRequest
from datadog_api_client.v2.model.spans_list_request_data import SpansListRequestData
from datadog_api_client.v2.model.spans_list_request_attributes import SpansListRequestAttributes
from datadog_api_client.v2.model.spans_list_request_page import SpansListRequestPage
from datadog_api_client.v2.model.spans_sort import SpansSort

REGION = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')
SSM_PREFIX = '/anycompany/agentcore'
ssm = boto3.client('ssm', region_name=REGION)

def get_ssm(n, d=None):
    try:
        return ssm.get_parameter(Name=n, WithDecryption=True)['Parameter']['Value']
    except Exception:
        if d is not None:
            return d
        raise

DD_API_KEY = os.environ.get('DD_API_KEY') or get_ssm(f'{SSM_PREFIX}/dd_api_key')
DD_APP_KEY = os.environ.get('DD_APP_KEY') or get_ssm(f'{SSM_PREFIX}/dd_app_key')
DD_SITE = os.environ.get('DD_SITE') or get_ssm(f'{SSM_PREFIX}/dd_site', 'datadoghq.com')

dd_config = Configuration()
dd_config.api_key['apiKeyAuth'] = DD_API_KEY
dd_config.api_key['appKeyAuth'] = DD_APP_KEY
dd_config.server_variables['site'] = DD_SITE

WINDOW_MIN = 60
DD_ENV = os.environ.get('DD_ENV', 'demo')  # matches DD_ENV on the L3 agent deploys
SERVICE_QUERY = 'service:anycompany-* -resource_name:"GET /ping"'

def fetch_spans(query, minutes=WINDOW_MIN, limit=1000):
    _now = datetime.now(timezone.utc)
    with ApiClient(dd_config) as client:
        body = SpansListRequest(data=SpansListRequestData(type='search_request',
            attributes=SpansListRequestAttributes(
                filter={'from': (_now - timedelta(minutes=minutes)).strftime('%Y-%m-%dT%H:%M:%SZ'),
                        'to': _now.strftime('%Y-%m-%dT%H:%M:%SZ'), 'query': query},
                sort=SpansSort.TIMESTAMP_DESCENDING, page=SpansListRequestPage(limit=limit))))
        return [s.get('attributes', {}) for s in SpansApi(client).list_spans(body=body).get('data', [])]

print('Datadog site :', DD_SITE)
print('Datadog env  :', DD_ENV)
print('Service scope:', SERVICE_QUERY)


## Step 2: Generate Sample Traffic

Invoke the Harness a couple of times so the Service Map and Error Tracking views have
fresh spans. Each run exercises the orchestrator + specialists.

In [ ]:
cognito = boto3.client('cognito-idp', region_name=REGION)
agentcore = boto3.client('bedrock-agentcore', region_name=REGION, config=Config(read_timeout=300))
cfg = {k: get_ssm(f'{SSM_PREFIX}/{k}') for k in ['harness_arn', 'cognito_client_id', 'user_password']}
cognito.initiate_auth(ClientId=cfg['cognito_client_id'], AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={'USERNAME': 'gold_customer', 'PASSWORD': cfg['user_password']})
print('Authenticated as gold_customer\n')

for p in ['What is the status of order ORD-10001?', 'Which of my orders are eligible for a refund?']:
    sid = str(uuid.uuid4()).upper()
    print('>>>', p)
    r = agentcore.invoke_harness(harnessArn=cfg['harness_arn'], runtimeSessionId=sid, actorId='CUST-789',
        messages=[{'role': 'user', 'content': [{'text': f'[Authenticated customer: CUST-789 | Tier: gold]\n\n{p}'}]}])
    out = ''.join(ev['contentBlockDelta'].get('delta', {}).get('text', '')
                  for ev in r['stream'] if 'contentBlockDelta' in ev)
    print((out[:160] + '...') if out else '(no text)', '\n')

print('Waiting 60s for spans to reach Datadog...')
time.sleep(60)
print('Done.')


## Step 3: Service Map - the Auto-Built Topology

Datadog assembles a **Service Map** from the spans - every service, plus the request
rate, latency, and error rate on each edge - with no wiring from you. Below we list the
services seen and any cross-service edges we can resolve from the spans, then link to the
full interactive map.

In the console the map lives under **Developer Portal > Catalog > Services**, `Map` view
(`/software?view=map`). Set the **Env** selector to `demo` — that is the `DD_ENV` the
agents deploy with.

On the **Map** you see the **two specialist services** (`anycompany-order-agent`,
`anycompany-refund-agent`) and their Gateway/tool dependencies. The **orchestrator does
not appear on the Map** — it registers in the Services **List** and in **Agent
Observability** (nb3), but not on the APM dependency Map. This is a property of *how* the
orchestrator is instrumented, not a bug: the specialists emit APM spans through the
ddtrace **sidecar** that builds the Map topology, while the Harness is AWS-managed and
exports traces via **OTLP-direct** — enough to create a Catalog entity and feed Agent
Observability, but it does not populate the APM dependency Map (this holds regardless of
the `dd-otlp-source` header). The printed span list above may still include
`anycompany-orchestrator`, because span *search* and the dependency *Map* are different
views of the data.

In [ ]:
spans = fetch_spans(SERVICE_QUERY)
by_id = {s.get('span_id'): s for s in spans}
services = sorted({s.get('service', '?') for s in spans})

# Best-effort dependency edges: a span whose parent belongs to a DIFFERENT service.
edges = set()
for s in spans:
    parent = by_id.get(s.get('parent_id'))
    if parent and parent.get('service') != s.get('service'):
        edges.add((parent.get('service'), s.get('service')))

print(f'Services seen in the last {WINDOW_MIN} min ({len(services)}):')
for svc in services:
    print('  -', svc)

print('\nCross-service edges resolved from spans:')
if edges:
    for a, b in sorted(edges):
        print(f'  {a} -> {b}')
else:
    print('  (none resolved here - the Harness (OTLP) and the sidecar agents are separate')
    print('   telemetry pipelines, so cross-pipeline parent links may not join in the raw')
    print('   spans. Datadog still infers the full topology in the UI using peer tags.)')

# Deep-link straight into the interactive map, scoped to the same window queried above.
_end_ms = int(time.time() * 1000)
_start_ms = _end_ms - WINDOW_MIN * 60 * 1000
print(f'\nOpen the interactive Service Map (topology + rate/latency/errors per edge):')
print(f'  https://app.{DD_SITE}/software?env={DD_ENV}&fromUser=true&view=map'
      f'&start={_start_ms}&end={_end_ms}')
print('  (In the console: Developer Portal > Catalog > Services, then the "Map" view.)')


### Service Map in Datadog

**Developer Portal > Catalog > Services**, `Map` view with **Env** = `demo`, draws the
topology from the spans: the **two specialist services** and their Gateway/tool
dependencies, with node outlines colored by service health. Click an edge to drill into
its traces. The orchestrator (Harness) is not on this Map — see the note above; its view
is in **Agent Observability** (nb3).

![Datadog Service Map](datadog_service_map.png)

## Step 4: Error Tracking - Group Failures into Issues

Datadog **Error Tracking** clusters error spans into *issues* (first seen, count,
affected services) so a recurring failure is one line, not thousands. Below we pull the
recent error spans and summarize them; open Error Tracking for the grouped view.

Good sources of errors in this workshop: the **AI Guard gateway 403s** (L4), harness
**Max-iterations / timeout** failures, and tool errors.

In [ ]:
errs = fetch_spans('service:anycompany-* status:error')
print(f'Error spans in the last {WINDOW_MIN} min: {len(errs)}\n')

if errs:
    by = defaultdict(int)
    for s in errs:
        by[(s.get('service', '?'), s.get('resource_name', '?'))] += 1
    print(f'{"Service":<26} {"Resource":<36} {"Count":>6}')
    print('-' * 72)
    for (svc, res), n in sorted(by.items(), key=lambda x: -x[1]):
        print(f'{svc[:25]:<26} {str(res)[:35]:<36} {n:>6}')
else:
    print('No error spans in the window. To generate some, run an L4 enforcement scenario -')
    print('e.g. trip the AI Guard gateway breaker (l4-security/3 or /5); those 403s appear here.')

print('\nDatadog groups these into issues (count, last seen, owning service).')
print('Open Errors > Error Tracking in the console, Issues tab.')


### Error Tracking in Datadog

**Errors > Error Tracking** (`Issues` tab) collapses the raw error spans above into
*issues* - each row is one recurring failure with its occurrence count, how long it has
been seen, and the service it came from. Filter by `Service` in the left facet panel to
narrow to the agent services.

![Datadog Error Tracking](datadog_error_tracking.png)

## Step 5: SLO - a Reliability Target

Define a **Service Level Objective**: the fraction of agent requests that did not error,
over a 7-day window, with a 99% target.

This is where the telemetry stops being descriptive and becomes a commitment. Steps 3 and
4 tell you what happened; an SLO states what you promised, and Datadog tracks an **error
budget** against it — at a 99% target over 7 days, 1% of requests may fail before you
breach. That budget is the number you actually manage: plenty left means you can ship and
tighten guardrails; nearly gone means stop and stabilize.

It complements the monitors from Notebook 2 rather than duplicating them. A monitor fires
on a threshold *right now* ("p95 latency > 60s"). An SLO accumulates over a window, so a
brief blip barely dents the budget while a slow steady error rate shows up clearly. That
distinction matters for agents, which fail probabilistically — a single bad generation is
noise, a drifting failure rate is a regression.

The SLO is built on the `anycompany.agent.requests` / `anycompany.agent.errors` custom
metrics **Notebook 2** submits, so run nb2 first.

Creating an SLO through the API needs the `slos_write` scope on your Application key (the
full scope list is in `l3-orchestration/1_setup_resources.ipynb` Step 6). If your key does
not carry it, the cell prints the SLO definition to paste into **Monitoring > SLOs > New
SLO > Metric** — the resulting SLO is identical either way.

In [ ]:
from datadog_api_client.v1.api.service_level_objectives_api import ServiceLevelObjectivesApi

SLO = {
    'type': 'metric',
    'name': '[AgentCore] Agent request success rate >= 99%',
    'description': 'Fraction of agent requests that did not error, across the workshop services.',
    'query': {
        'numerator': 'sum:anycompany.agent.requests{*}.as_count() - sum:anycompany.agent.errors{*}.as_count()',
        'denominator': 'sum:anycompany.agent.requests{*}.as_count()',
    },
    'thresholds': [{'timeframe': '7d', 'target': 99.0, 'warning': 99.5}],
    'tags': ['team:agentcore', 'env:workshop'],
}

try:
    with ApiClient(dd_config) as client:
        resp = ServiceLevelObjectivesApi(client).create_slo(body=SLO)
        sid = resp.data[0].id if getattr(resp, 'data', None) else '?'
        print(f'Created SLO (id: {sid})')
        print(f'  View: https://app.{DD_SITE}/slo/manage')
except Exception as e:
    if 'Forbidden' in str(e) or '403' in str(e):
        print('Your app key lacks the `slos_write` scope - SLO not created via API.')
        print('Create it in the UI: Monitoring > SLOs > New SLO > Metric, using the')
        print('definition below. To use the API instead, add `slos_write` to the key')
        print('(scope list: l3-orchestration/1_setup_resources.ipynb Step 6).\n')
        print(json.dumps(SLO, indent=2))
    else:
        print(f'SLO create failed: {e}')
        print('(The SLO uses nb2 custom metrics - run Notebook 2 first if they are missing.)')


### SLO in Datadog

**Monitoring > SLOs** lists the target with its live **status** and **error budget left** -
how much failure you can still absorb in the 7-day window before you breach 99%.

![Datadog SLO](datadog_slo.png)

## Summary

| Feature | What it gives you | How |
|---|---|---|
| Service Map | Auto topology of the multi-agent system with per-edge rate/latency/errors | Derived by Datadog from spans |
| Error Tracking | Failures grouped into issues (403s, timeouts, tool errors) | Derived from error spans |
| SLO | A 99%/7d reliability commitment plus the error budget you manage against it | `anycompany.*` custom metrics (from nb2) |

### L6 Complete

| Notebook | Coverage |
|----------|----------|
| 1 - Multi-Agent Tracing | Distributed APM traces across services |
| 2 - Metrics & Cost | Span-derived metrics, token cost, monitors, dashboard |
| 3 - Agent Observability | AI-specific Agent / LLM / Tool spans |
| 4 - Quality Evaluations | LLM-as-judge + deterministic + managed evaluations |
| 5 - Service Map, Errors & SLOs | Topology, grouped errors, reliability targets |


## Cleanup

Service Map and Error Tracking are **auto-derived from spans** - there is nothing to
delete. The SLO from Step 5 is the one object this notebook creates - remove it below
(needs `slos_write`; otherwise delete it from **Monitoring > SLOs**).

In [ ]:
# === CLEANUP: uncomment to delete the SLO created in Step 5 ===
# from datadog_api_client.v1.api.service_level_objectives_api import ServiceLevelObjectivesApi
# with ApiClient(dd_config) as client:
#     api = ServiceLevelObjectivesApi(client)
#     found = api.list_slos(tags_query='team:agentcore').get('data', [])
#     for slo in found:
#         try:
#             api.delete_slo(slo_id=slo['id'])
#             print('Deleted SLO:', slo.get('name'))
#         except Exception as e:
#             print('Delete failed:', e)
# print('Service Map + Error Tracking are auto-derived - nothing else to clean up.')
